# Notebook 08 — Packing, Long Context, and Throughput

On Colab Pro, throughput often matters more than optimizer folklore. If you waste tokens on padding, choose an oversized context length, or ignore gradient accumulation, you can make every experiment slower without improving the model.

## What you will build

This notebook studies throughput from first principles:

1. inspect token-length distributions
2. measure padding waste with and without packing
3. compare realistic max-sequence choices
4. translate gradient accumulation into effective tokens per optimizer step
5. turn the results into Colab Pro batch-strategy defaults

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Step 1: Define notebook paths and helper functions

The helper functions stay intentionally simple so you can see what sequence packing is trying to optimize.

In [ ]:
random.seed(8)

ARTIFACT_DIR = Path(OUTPUT_DIR) / "notebook_08_packing_and_throughput"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def render_chat(record):
    messages = [
        {"role": "system", "content": "You are Castalia Mentor, a concise tutor who answers with practical finetuning advice."},
        {"role": "user", "content": record["user"]},
        {"role": "assistant", "content": record["assistant"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def greedy_pack(lengths, max_seq_length):
    bins = []
    for length in sorted(lengths, reverse=True):
        placed = False
        for packed_bin in bins:
            if packed_bin["used_tokens"] + length <= max_seq_length:
                packed_bin["used_tokens"] += length
                packed_bin["items"] += 1
                placed = True
                break
        if not placed:
            bins.append({"used_tokens": length, "items": 1})
    return bins

def summarize_no_packing(lengths, max_seq_length):
    clipped = [min(length, max_seq_length) for length in lengths]
    wasted = [max_seq_length - value for value in clipped]
    return {
        "mode": "no_packing",
        "max_seq_length": max_seq_length,
        "examples_or_bins": len(clipped),
        "mean_utilization": round(sum(clipped) / (len(clipped) * max_seq_length), 3),
        "mean_wasted_tokens": round(sum(wasted) / len(wasted), 1),
        "truncated_examples": int(sum(length > max_seq_length for length in lengths)),
    }

def summarize_packing(lengths, max_seq_length):
    clipped = [min(length, max_seq_length) for length in lengths]
    packed_bins = greedy_pack(clipped, max_seq_length)
    wasted = [max_seq_length - packed_bin["used_tokens"] for packed_bin in packed_bins]
    return {
        "mode": "packing",
        "max_seq_length": max_seq_length,
        "examples_or_bins": len(packed_bins),
        "mean_utilization": round(sum(packed_bin["used_tokens"] for packed_bin in packed_bins) / (len(packed_bins) * max_seq_length), 3),
        "mean_wasted_tokens": round(sum(wasted) / len(wasted), 1),
        "truncated_examples": int(sum(length > max_seq_length for length in lengths)),
    }

print("Artifacts:", ARTIFACT_DIR.resolve())

## Step 2: Build a variable-length local dataset

Packing only matters when lengths vary enough that padding waste becomes expensive. So we intentionally mix short, medium, and longer examples.

In [ ]:
short_context = "Short context: a student wants one crisp answer about finetuning basics. "
medium_context = "Medium context: the student shares setup details, asks for trade-offs, and wants one practical example. "
long_context = "Long context: the student pastes experiment notes, benchmark comments, and failed-run observations. "

records = []
for index, repeat_count in enumerate([3, 4, 5, 6, 8, 10, 12, 16, 20, 28, 36, 44], start=1):
    if repeat_count <= 6:
        base_context = short_context
        topic = "short"
    elif repeat_count <= 16:
        base_context = medium_context
        topic = "medium"
    else:
        base_context = long_context
        topic = "long"

    user_text = (
        "Summarize the throughput trade-off for this training setup. "
        + base_context * repeat_count
        + "Focus on whether packing, shorter context, or more gradient accumulation is the first lever."
    )
    assistant_text = (
        "Start by measuring token lengths and padding waste. "
        "Packing helps when many examples are short, shorter max_seq_length helps when truncation stays acceptable, "
        "and gradient accumulation helps when per-device batch size must stay tiny."
    )
    records.append({"id": f"ex_{index:02d}", "topic": topic, "user": user_text, "assistant": assistant_text})

length_df = pd.DataFrame(records)
length_df["text"] = length_df.apply(render_chat, axis=1)
length_df["token_length"] = length_df["text"].map(lambda text: len(tokenizer(text, add_special_tokens=False)["input_ids"]))

display(length_df[["id", "topic", "token_length"]])
print("Examples:", len(length_df))

## Step 3: Inspect basic length statistics

Never pick a context length by instinct. First check the actual distribution.

In [ ]:
length_stats_df = pd.DataFrame(
    [
        {"metric": "min_tokens", "value": int(length_df["token_length"].min())},
        {"metric": "median_tokens", "value": int(length_df["token_length"].median())},
        {"metric": "mean_tokens", "value": round(float(length_df["token_length"].mean()), 1)},
        {"metric": "p90_tokens", "value": round(float(length_df["token_length"].quantile(0.9)), 1)},
        {"metric": "max_tokens", "value": int(length_df["token_length"].max())},
    ]
)
display(length_stats_df)

## Step 4: Visualize the token-length distribution

The shape matters. A strongly mixed distribution usually means packing will pay off.

In [ ]:
distribution_df = length_df[["id", "topic", "token_length"]].sort_values("token_length").reset_index(drop=True)
display(distribution_df)

ax = distribution_df.plot(
    y="token_length",
    kind="hist",
    bins=8,
    figsize=(7, 4),
    title="Token length distribution",
    color="tab:blue",
)
ax.set_xlabel("tokens")
plt.tight_layout()

## Step 5: Compute utilization without packing

Without packing, each example occupies a full max-sequence slot whether it needs the space or not.

In [ ]:
lengths = distribution_df["token_length"].tolist()
no_packing_df = pd.DataFrame(
    [summarize_no_packing(lengths, max_seq_length) for max_seq_length in [512, 1024, 2048, 4096]]
)
display(no_packing_df)

## Step 6: Compute utilization with packing

Packing tries to combine shorter examples into fuller training sequences. That reduces padding waste and can raise effective throughput.

In [ ]:
packing_df = pd.DataFrame(
    [summarize_packing(lengths, max_seq_length) for max_seq_length in [512, 1024, 2048, 4096]]
)
display(packing_df)

## Step 7: Merge the two views into one comparison table

This is the first core table of the notebook.

In [ ]:
comparison_df = no_packing_df.merge(
    packing_df,
    on="max_seq_length",
    suffixes=("_no_packing", "_packing"),
)
comparison_df["utilization_gain"] = (comparison_df["mean_utilization_packing"] - comparison_df["mean_utilization_no_packing"]).round(3)
comparison_df["wasted_token_reduction"] = (comparison_df["mean_wasted_tokens_no_packing"] - comparison_df["mean_wasted_tokens_packing"]).round(1)
comparison_df.to_csv(ARTIFACT_DIR / "packing_comparison.csv", index=False)

display(
    comparison_df[
        [
            "max_seq_length",
            "mean_utilization_no_packing",
            "mean_utilization_packing",
            "utilization_gain",
            "wasted_token_reduction",
            "truncated_examples_no_packing",
        ]
    ]
)

## Step 8: Visualize padding waste directly

If the two bars are very different, packing is probably your first throughput lever.

In [ ]:
plot_df = comparison_df[["max_seq_length", "mean_wasted_tokens_no_packing", "mean_wasted_tokens_packing"]].copy()
plot_df = plot_df.set_index("max_seq_length")
ax = plot_df.plot(kind="bar", figsize=(8, 4), title="Average wasted tokens per training sequence")
ax.set_ylabel("tokens")
plt.xticks(rotation=0)
plt.tight_layout()

## Step 9: Measure how many examples survive each max-sequence choice

Longer context only helps if shorter limits are truncating useful data.

In [ ]:
max_length_rows = []
for max_seq_length in [512, 1024, 1536, 2048, 3072, 4096]:
    retained_fraction = round(sum(length <= max_seq_length for length in lengths) / len(lengths), 3)
    truncated_fraction = round(sum(length > max_seq_length for length in lengths) / len(lengths), 3)
    max_length_rows.append(
        {
            "max_seq_length": max_seq_length,
            "retained_fraction": retained_fraction,
            "truncated_fraction": truncated_fraction,
            "all_examples_fit": bool(max(lengths) <= max_seq_length),
        }
    )

max_length_df = pd.DataFrame(max_length_rows)
display(max_length_df)

## Step 10: Build a simple context-choice rubric

This table helps decide when a bigger max sequence length is justified and when it merely slows training.

In [ ]:
context_rubric_df = max_length_df.copy()
context_rubric_df["recommended_read"] = context_rubric_df.apply(
    lambda row: (
        "strong candidate"
        if row["truncated_fraction"] <= 0.1
        else "maybe if those clipped examples matter"
        if row["truncated_fraction"] <= 0.3
        else "too much clipping"
    ),
    axis=1,
)
display(context_rubric_df)

## Step 11: Translate gradient accumulation into effective tokens per optimizer step

When VRAM limits per-device batch size, gradient accumulation becomes the main batch-size lever.

In [ ]:
accumulation_configs = [
    {"name": "t4_safe", "gpu": "T4", "per_device_batch": 1, "grad_accum": 8, "max_seq_length": 1024},
    {"name": "t4_longer_context", "gpu": "T4", "per_device_batch": 1, "grad_accum": 8, "max_seq_length": 2048},
    {"name": "l4_balanced", "gpu": "L4", "per_device_batch": 2, "grad_accum": 4, "max_seq_length": 2048},
    {"name": "l4_long_context", "gpu": "L4", "per_device_batch": 1, "grad_accum": 4, "max_seq_length": 4096},
]

accumulation_df = pd.DataFrame(accumulation_configs)
accumulation_df["packing_utilization"] = accumulation_df["max_seq_length"].map(
    lambda max_seq_length: float(comparison_df.loc[comparison_df["max_seq_length"] == max_seq_length, "mean_utilization_packing"].iloc[0])
)
accumulation_df["effective_tokens_per_update"] = (
    accumulation_df["per_device_batch"]
    * accumulation_df["grad_accum"]
    * accumulation_df["max_seq_length"]
    * accumulation_df["packing_utilization"]
).round(0).astype(int)
display(accumulation_df)

## Step 12: Add a teaching-level throughput model

These are not universal constants. They are deliberately simple assumptions to reason about trade-offs across common Colab GPUs.

In [ ]:
throughput_assumptions = {
    "T4": {512: 7000, 1024: 4700, 2048: 2500, 4096: 1200},
    "L4": {512: 12000, 1024: 8200, 2048: 4700, 4096: 2400},
}

scenario_rows = []
for _, row in accumulation_df.iterrows():
    tokens_per_second = throughput_assumptions[row["gpu"]][int(row["max_seq_length"])]
    scenario_rows.append(
        {
            "gpu": row["gpu"],
            "config": row["name"],
            "max_seq_length": int(row["max_seq_length"]),
            "effective_tokens_per_update": int(row["effective_tokens_per_update"]),
            "estimated_tokens_per_second": tokens_per_second,
            "estimated_seconds_per_update": round(row["effective_tokens_per_update"] / tokens_per_second, 2),
        }
    )

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

## Step 13: Compare short-context versus long-context batch strategy

This is where the main Colab trade-off appears: longer context raises coverage but often cuts tokens-per-second hard.

In [ ]:
strategy_compare_df = scenario_df.pivot(index="gpu", columns="config", values="estimated_seconds_per_update").reset_index()
display(strategy_compare_df)

## Step 14: Run a tiny forward-pass throughput probe on the current runtime

This is not a full training benchmark. It is just a quick way to show how sequence length alone changes observed tokens-per-second.

In [ ]:
model.eval()
benchmark_rows = []
benchmark_prompt = "Summarize the main throughput lever for this setup. " + ("Context note. " * 400)

for seq_length in [512, 1024, 2048]:
    encoded = tokenizer(
        benchmark_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=seq_length,
    )
    input_ids = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    _ = model(input_ids=input_ids, attention_mask=attention_mask)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(3):
        _ = model(input_ids=input_ids, attention_mask=attention_mask)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start_time

    processed_tokens = int(input_ids.shape[-1] * 3)
    benchmark_rows.append(
        {
            "seq_length": int(input_ids.shape[-1]),
            "processed_tokens": processed_tokens,
            "elapsed_s": round(elapsed, 3),
            "observed_tokens_per_second": round(processed_tokens / elapsed, 1),
        }
    )

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(ARTIFACT_DIR / "forward_pass_benchmark.csv", index=False)
display(benchmark_df)

## Step 15: Visualize the observed throughput drop

Even a tiny probe usually shows the basic pattern: tokens per second fall as sequence length rises.

In [ ]:
ax = benchmark_df.plot(
    kind="line",
    x="seq_length",
    y="observed_tokens_per_second",
    marker="o",
    figsize=(7, 4),
    title="Observed tokens per second by sequence length",
)
ax.set_ylabel("tokens / second")
plt.tight_layout()

## Step 16: Show why gradient accumulation is a batch-shape tool, not free compute

Accumulation lets you keep more tokens in each optimizer update, but it does not remove the underlying cost of longer sequences.

In [ ]:
accumulation_explainer_df = accumulation_df.copy()
accumulation_explainer_df["tokens_per_microbatch"] = (
    accumulation_explainer_df["per_device_batch"]
    * accumulation_explainer_df["max_seq_length"]
    * accumulation_explainer_df["packing_utilization"]
).round(0).astype(int)
display(accumulation_explainer_df[["name", "gpu", "tokens_per_microbatch", "grad_accum", "effective_tokens_per_update"]])

## Step 17: Simulate an already-long dataset

Packing is not always the hero. If almost every example is already long, there is much less padding waste to remove.

In [ ]:
long_heavy_lengths = [1800, 1900, 2000, 2048, 2048, 2048, 1900, 1750]
long_heavy_df = pd.DataFrame(
    [
        summarize_no_packing(long_heavy_lengths, 2048),
        summarize_packing(long_heavy_lengths, 2048),
    ]
)
display(long_heavy_df)

## Step 18: Summarize when packing helps most

This turns the raw calculations into a reusable rule of thumb.

In [ ]:
packing_rules_df = pd.DataFrame(
    [
        {"dataset_shape": "mostly short", "packing_value": "very high", "reason": "padding waste dominates"},
        {"dataset_shape": "mixed short and medium", "packing_value": "high", "reason": "short examples can share sequences"},
        {"dataset_shape": "mostly long", "packing_value": "limited", "reason": "there is little empty space to reclaim"},
        {"dataset_shape": "nearly uniform length", "packing_value": "modest", "reason": "padding waste is already controlled"},
    ]
)
display(packing_rules_df)

## Step 19: Create a Colab Pro batch-strategy table

This is the practical output most teams actually need.

In [ ]:
batch_strategy_df = pd.DataFrame(
    [
        {
            "scenario": "T4 with mixed short and medium data",
            "recommended_batch_strategy": "packing on, per_device_batch_size 1, grad_accum 8, max_seq_length 1024 or 2048",
            "why": "best balance of utilization and runtime safety",
        },
        {
            "scenario": "T4 with genuinely long examples",
            "recommended_batch_strategy": "packing if lengths vary, otherwise keep batch tiny and test 2048 before 4096",
            "why": "long context is expensive, so justify it with actual truncation pressure",
        },
        {
            "scenario": "L4 with balanced SFT run",
            "recommended_batch_strategy": "packing on, per_device_batch_size 2, grad_accum 4, max_seq_length 2048",
            "why": "more room for both batch size and useful context",
        },
        {
            "scenario": "L4 with long-context task",
            "recommended_batch_strategy": "start at 2048, move to 4096 only if clipped examples matter",
            "why": "preserve throughput unless the task clearly needs the extra window",
        },
    ]
)
display(batch_strategy_df)

## Step 20: Add a long-context checklist

Long context is valuable when the task requires it, not because a larger number feels more powerful.

In [ ]:
long_context_checklist_df = pd.DataFrame(
    [
        {"question": "Do many examples exceed the current max_seq_length?", "action": "measure token lengths first"},
        {"question": "Is truncation dropping task-relevant information?", "action": "inspect clipped examples manually"},
        {"question": "Is throughput already the bottleneck?", "action": "benchmark before increasing context"},
        {"question": "Would packing solve the main waste instead?", "action": "try packing before increasing length"},
    ]
)
display(long_context_checklist_df)

## Step 21: Pick the notebook default by scenario

The right answer depends on dataset shape and GPU, not on a universal magic number.

In [ ]:
scenario_defaults = {
    "mixed_dataset_t4": "packing on with 1024 or 2048, plus gradient accumulation",
    "mostly_short_dataset": "packing is the first lever",
    "long_context_dataset_l4": "use 2048 first and justify 4096 with truncation evidence",
}

scenario_defaults

## Step 22: Save a compact throughput manifest

This makes the notebook output easy to hand off to the next training run.

In [ ]:
throughput_manifest = {
    "best_first_lever_for_mixed_lengths": "packing",
    "context_choice_should_follow_data": True,
    "gradient_accumulation_is_batch_shape_not_magic": True,
    "recommended_t4_strategy": batch_strategy_df.iloc[0]["recommended_batch_strategy"],
}

with open(ARTIFACT_DIR / "throughput_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(throughput_manifest, handle, indent=2)

throughput_manifest

## Key takeaways

- Packing is often the easiest throughput win on mixed-length datasets.
- Max sequence length should be chosen from measured truncation pressure, not instinct.
- Gradient accumulation changes effective batch size when VRAM keeps microbatches small.
- Longer context is only worth its cost when the data truly needs it.
- A good Colab Pro batch strategy balances utilization, truncation risk, and iteration speed.